<a href="https://colab.research.google.com/github/JavierSanLorenzoVIU/03MAIR---Algoritmos-de-Optimizacion---2026/blob/main/TrabajoPractico/Trabajo_Pr%C3%A1ctico_Algoritmos_JavierSanLorenzoGomez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Javier San Lorenzo Gómez  <br>
Url: https://github.com/JavierSanLorenzoVIU/03MAIR---Algoritmos-de-Optimizacion---2026/tree/main/TrabajoPractico<br>
Google Colab: https://colab.research.google.com/drive/1FWuczaLsRA8_dXKAh9zcPk_gM-fIqdMw#scrollTo=96W1V2quy3QK <br>


Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de una jornada de La Liga<br>
>3. Configuración de Tribunales

Descripción del problema:

---

###**Problema 2**

Organizar los horarios de partidos de La Liga(I)
- Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma que maximice la audiencia.


- Los horarios disponibles se conocen a priori y son los siguientes:

| Viernes |     Sábado    |    Domingo    | Lunes |
|---------|---------------|---------------|-------|
|   20    | 12, 16, 18, 20| 12, 16, 18, 20|   20  |


- En primer lugar se clasifican los equipos en tres categorías según el numero de seguidores( que tiene relación directa con la audiencia). Hay 3 equipos en la categoría A, 11 equipos de categoría B y 6 equipos de categoría C.


- Se conoce estadísticamente la audiencia que genera cada partido según los equipos que se enfrentan y en horario de sábado a las 20h (el mejor en todos los casos)

|     .     |  Categoría A  |  Categoría B  |  Categoría C  |
|-----------|---------------|---------------|---------------|
|Categoría A|   2 Millones  | 1,3 Millones  | 1 Millón      |
|Categoría B|               | 0,9 Millones  | 0,75 Millones |
|Categoría C|               |               | 0,47 Millones |

- Si el horario del partido no se realiza a las 20 horas del sábado se sabe que se reduce según los coeficientes de la siguiente tabla

- Debemos asignar obligatoriamente siempre un partido el viernes y un partido el lunes

| . | Viernes |  Sábado |  Domingo   | Lunes |
|---|---------|---------|------------|-------|
|12h|  -  | 0.55  | 0.45 |  -  |
|16h|  -  | 0.7   | 0.75 |  -  |
|18h|  -  | 0.8   | 0.85 |  -  |
|20h| 0.4 | 1     |  1   | 0.4 |


- Es posible la coincidencia de horarios pero en este caso la audiencia de cada partido se verá afectada y se estima que se reduce en porcentaje según la siguiente tabla dependiendo del número de coincidencias:

| Coincidencias | -% |
|---|---------|
|0|  0%  |
|1|  25% |
|2|  45% |
|3|  60% |
|4|  70% |
|5|  75% |
|6|  78% |
|7|  80% |
|8|  80% |

- Los cálculos asociados a una jornada de ejemplo se realizan según se muestra en la siguiente tabla:

| Partido | Categorías |  Horario |  Base(Mill.)   | Ponderación |  Base x Ponderación |  Corrección Coincidencia |
|---|---------|---------|------------|-------|------------|-------|
|Celta - Real Madrid|  B-A  | V20  | 1,3 |  0,4  | 0,52 |  0,52  |
|Valencia - Real Sociedad|  B-A  | S12  | 1,3 |  0,55  | 0,72 |  0,72  |
|Mallorca - Eibar|  C-C  | S16  | 1,3 |  0,7  | 0,33 |  0,33  |
|Athletic - Barcelona|  B-A  | S18  | 1,3 |  0,8  | 1,04 |  1,04  |
|Leganés -  Osasuna|  C-C  | S20  | 0,47 |  1  | 0,47 |  0,47  |
|Villareal - Granada|  B-C  | D16  | 0,75 |  0,75  | 0,56 |  0,42  |
|Alavés - Levante|  B-B  | D16  | 0,9 |  0,75  | 0,68 |  0,51  |
|Espanyol - Sevilla|  B-B  | D18  | 0,9 |  0,85  | 0,77 |  0,7  |
|Betis - Valladolid|  B-C  | D20  | 0,75 |  1  | 0,75 |  0,75  |
|Atlético - Getafe|  B-B  | L20  | 0,9 |  0,4  | 0,36 |  0,36  |

---




In [ ]:
import pandas as pd
import numpy as np
import random

---
#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

###Representación del espacio de soluciones:

El conjunto de soluciones abarca todas las combinaciones posibles de emparejamientos entre los 20 equipos disponibles para formar los 10 partidos de la jornada, multiplicadas por todas las asignaciones potenciales de estos 10 partidos en las 10 franjas horarias disponibles.

La magnitud matemática de este problema es colosal. Por un lado, el número de formas distintas de emparejar a los 20 equipos en 10 partidos se calcula mediante combinatoria ( $\frac{20!}{2^{10} \cdot 10!}$), lo que genera 654.729.075 configuraciones de jornadas diferentes. Por otro lado, dado que las reglas permiten la coincidencia de horarios (solapamientos con penalización), cada uno de los 10 partidos dispone de 10 franjas horarias independientes, lo que supone 10<sup>10</sup> (10.000 millones) de asignaciones posibles por jornada. Al multiplicar ambos factores, el espacio de búsqueda total asciende a más de 6,54 trillones de soluciones posibles (6.54×10<sup>18</sup>).

Esta inmensidad combinatoria hace que el problema sea computacionalmente inabordable mediante métodos exactos o de fuerza bruta, justificando plenamente el desarrollo de una metaheurística avanzada (GRASP) para encontrar una solución de altísima calidad (cuasi-óptima).



Para representar el espacio de soluciones computacionalmente se emplean tuplas del tipo (*Equipo_Local*, *Equipo_Visitante*, *Id_Horario*).
Para representar los datos y resultados se emplearán DataFrames de Pandas.


El **enfoque del modelo** se basa en desarrollar una metaheurística GRASP (Procedimiento de Búsqueda Adaptativa Aleatoria Golosa). La base teórica parte de un algoritmo voraz (greedy), cuya estrategia implica tomar decisiones en cada etapa para construir una solución óptima en ese momento.

La agresividad de los algoritmos puramente voraces los hace eficientes, pero en este problema de horarios presentan un fallo crítico: la "trampa voraz". Si el algoritmo siempre toma la mejor decisión inmediata (emparejar a los equipos medios entre sí para ganar audiencia rápida), agota los equipos buenos y condena a los equipos de menor categoría a jugar entre ellos al final, cayendo en un óptimo local muy deficiente.

Para solucionar esto, nuestro enfoque GRASP añade dos fases vitales :

1. **Fase Constructiva Aleatorizada:** En lugar de elegir siempre la mejor opción absoluta, evalúa globalmente todos los cruces y horarios, elabora una Lista Restringida de Candidatos (RCL) con las mejores opciones y elige una al azar. Esto introduce la suficiente variabilidad para explorar caminos que un algoritmo voraz descartaría.

2. **Fase de Búsqueda Local:** Una vez construida la jornada, el algoritmo realiza movimientos iterativos (intercambiando horarios entre partidos o deshaciendo emparejamientos de equipos) hasta garantizar que no existe ninguna configuración vecina que mejore la audiencia, alcanzando un óptimo local muy robusto.

Además, para implementar este algoritmo se necesitará definir la función objetivo.

###Función Objetivo:

La función objetivo se encarga de calcular el valor que se busca maximizar o minimizar en el problema de optimización. En este caso: maximizar la audiencia total de la jornada de partidos considerando las categorías de los equipos, las franjas horarias y las penalizaciones por coincidencias horarias.


$$AudienciaTotal = \sum_{i=1}^{10} \left( AudienciaBase_i \times PonderacionHorario_i \times (1 - ReduccionCoincidencia_i) \right)$$

- ***10*** → número de partidos de la jornada (derivado de los 20 equipos disponibles).
- ***AudienciaBase<sub>i</sub>*** → audiencia del partido *i* expresada en millones. Su valor depende exclusivamente del cruce de las categorías (A, B o C) de los dos equipos que se enfrentan (por ejemplo, A vs B = 1.3 Millones)

- ***PonderacionHorario<sub>i</sub>*** → coeficiente de ponderación correspondiente al horario en el que se juega el partido *i* (con valores que van desde 0.4 para el viernes a las 20h, hasta 1 para el sábado y domingo a las 20h).

- ***ReduccionCoincidencia<sub>i</sub>*** → porcentaje de reducción de audiencia aplicado al partido *i* si coincide en la misma franja horaria con otros partidos (varia desde 0% si no hay coincidencias, hasta un 80% si coinciden 8 partidos).

###Restricciones:

- **Unicidad de participación**. Cada uno de los 20 equipos solo puede jugar un partido por jornada.

- **Obligatoriedad de franjas horarias**. El calendario final debe contar obligatoriamente con, al menos:
  - Un partido el viernes (V20).
  - Un partido el lunes (L20).

[Coincidencias horarias → Se permite asignar múltiples partidos al mismo horario, pero esto conlleva una reducción de la audiencia total.]


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

Los datos se componen de:
1. Un data frame con 30 equipos y 3 categorías, divididos de la siguiente manera:
    - 3 equipos en la Categoría A
    - 11 equipos en la Categoría B
    - 6 equipos en la Categoría C

In [ ]:
equipos_data = {
    'Equipo': [
        'Real Madrid', 'R. Sociedad', 'Barcelona',  # 3 equipos Categoría A
        'Celta', 'Valencia', 'Athletic', 'Villarreal', 'Alavés', 'Levante', 'Espanyol', 'Sevilla', 'Betis', 'Atlético', 'Getafe',  # 11 equipos Categoría B
        'Mallorca', 'Eibar', 'Leganés', 'Osasuna', 'Granada', 'Valladolid'  # 6 equipos Categoría C
    ],
    'Categoria': [
        'A', 'A', 'A',
        'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
        'C', 'C', 'C', 'C', 'C', 'C'
    ]
}
df_equipos = pd.DataFrame(equipos_data)

display(df_equipos)

,Equipo,Categoria
0,Real Madrid,A
1,R. Sociedad,A
2,Barcelona,A
3,Celta,B
4,Valencia,B
5,Athletic,B
6,Villarreal,B
7,Alavés,B
8,Levante,B
9,Espanyol,B


2. Un data frame con las audiencias base fijadas para cada tipo de enfrentamiento según las categorías de los equipos (A vs A, B vs C, etc.).



In [ ]:
audiencia_base_data = {
    'Categoria_1': ['A', 'A', 'A', 'B', 'B', 'C'],
    'Categoria_2': ['A', 'B', 'C', 'B', 'C', 'C'],
    'Audiencia_Base_Millones': [2.0, 1.3, 1.0, 0.9, 0.75, 0.47]
}
df_audiencia_base = pd.DataFrame(audiencia_base_data)

### Enfrentamiento C-C se extrae de la tabla 'Los cálculos asociados a una jornada de ejemplo' y se observa que es 0.47, parece que la tabla anterior aparece recortada y no se observa esa última fila ###

display(df_audiencia_base)

,Categoria_1,Categoria_2,Audiencia_Base_Millones
0,A,A,2.00
1,A,B,1.30
2,A,C,1.00
3,B,B,0.90
4,B,C,0.75
5,C,C,0.47


3. Un data frame con los horarios que hay disponibles para los partidos (con su correspondiente coeficiente de ponderación).

In [ ]:
horarios_data = {
    'Horario': ['V20', 'S12', 'S16', 'S18', 'S20', 'D12', 'D16', 'D18', 'D20', 'L20'],
    'Dia': ['Viernes', 'Sábado', 'Sábado', 'Sábado', 'Sábado', 'Domingo', 'Domingo', 'Domingo', 'Domingo', 'Lunes'],
    'Hora': ['20h', '12h', '16h', '18h', '20h', '12h', '16h', '18h', '20h', '20h'],
    'Ponderacion': [0.4, 0.55, 0.7, 0.8, 1.0, 0.45, 0.75, 0.85, 1.0, 0.4]
}
df_horarios = pd.DataFrame(horarios_data)

display(df_horarios)

,Horario,Dia,Hora,Ponderacion
0,V20,Viernes,20h,0.40
1,S12,Sábado,12h,0.55
2,S16,Sábado,16h,0.70
3,S18,Sábado,18h,0.80
4,S20,Sábado,20h,1.00
5,D12,Domingo,12h,0.45
6,D16,Domingo,16h,0.75
7,D18,Domingo,18h,0.85
8,D20,Domingo,20h,1.00
9,L20,Lunes,20h,0.40


4. Un data frame con los coeficientes de reducción de audiencia al coincidir varios partidos a la misma hora (con su correspondiente coeficiente de reducción de audiencia).

In [ ]:
coincidencias_data = {
    'Coincidencias': [0, 1, 2, 3, 4, 5, 6, 7, 8],
    'Reduccion_Porcentaje': [0.0, 0.25, 0.45, 0.60, 0.70, 0.75, 0.78, 0.80, 0.80]
}
df_coincidencias = pd.DataFrame(coincidencias_data)

display(df_coincidencias)

,Coincidencias,Reduccion_Porcentaje
0,0,0.00
1,1,0.25
2,2,0.45
3,3,0.60
4,4,0.70
5,5,0.75
6,6,0.78
7,7,0.80
8,8,0.80


###Complejidad y Enfoque:

- **Método**: Se emplea la metaheurística GRASP (fase constructiva voraz optimizada + búsqueda local).

- **Complejidad**: Resolverlo por fuerza bruta implicaría una complejidad exponencial inasumible computacionalmente. Al aplicar GRASP, el problema se reduce a un tiempo polinómico (aprox. *O(n<sup>3</sup>)* ), proporcionando una excelente aproximación a la solución óptima en segundos al explorar solo los vecindarios más prometedores.

###Contabilización del espacio de soluciones:
Para 20 equipos (10 partidos), el espacio exploratorio se divide en dos dimensiones:

  - **Emparejamientos**: Las combinaciones para cruzar a los equipos se calculan mediante  $\frac{20!}{2^{10} \cdot 10!}$, generando 654.729.075 posibles jornadas.

  - **Horarios**: De las 10<sup>10</sup> asignaciones posibles (admitiendo solapamientos), quedan:
  
    10<sup>10</sup> −9<sup>10</sup>*[No viernes]* −9<sup>10</sup>*[No lunes]* +8<sup>10</sup>*[Solapamiento (Ni viernes ni lunes)]* = 4.100.173.022 distribuciones válidas tras aplicar las restricciones de jugar obligatoriamente el viernes y el lunes.

- **Total Factible**: Al multiplicar ambas magnitudes, el espacio real de soluciones válidas asciende a 2.68×10<sup>18</sup> combinaciones.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

En cuanto al diseño, se ha empleado la metaheurística **GRASP** (Greedy Randomized Adaptive Search Procedure), estructurada en dos etapas: una **Fase Constructiva global** y una **Fase de Búsqueda Local** con vecindario avanzado.

---

Un algoritmo puramente voraz (Greedy) fracasa en este problema porque cae en la "trampa voraz": empareja rápidamente a los equipos medios para ganar audiencia inmediata, agotando las opciones y obligando a los peores equipos a jugar entre sí al final, lo que hunde la audiencia total. GRASP soluciona esto dividiendo el trabajo:

1. **Fase Constructiva (RCL = 5)**: En lugar de coger siempre la mejor opción ciega, el algoritmo evalúa todos los cruces globales, coge los 5 mejores (Lista Restringida de Candidatos) y elige uno al azar. Esta pequeña flexibilidad permite al algoritmo hacer "sacrificios" a corto plazo para cuadrar una jornada perfecta a largo plazo.
2. **Búsqueda Local**: Refina la solución inicial mediante tres movimientos:
    - Intercambiar horarios entre dos partidos.
    - Mover un partido a una franja libre.
    - Cruzar equipos entre partidos distintos: (Movimiento clave que permite al algoritmo descubrir y formar los emparejamientos más rentables, como el A vs A).

El diseño de este algoritmo sigue los cinco componentes fundamentales que dan nombre a GRASP (Greedy Randomized Adaptive Search Procedure). A continuación, se detalla cómo se han conseguido implementar cada uno de ellos:

1. **Voracidad (Greedy)**
En la función fase_constructiva_global (o su versión para DataFrames), la voracidad se logra evaluando el beneficio inmediato de todas las combinaciones posibles en cada paso. Mediante bucles anidados, el algoritmo cruza todos los equipos disponibles (equipos_disp) en las 10 franjas horarias y calcula el beneficio esperado. Luego, ordena esta lista de candidatos de mayor a menor audiencia con la instrucción candidatos.sort(key=lambda x: x['Beneficio'], reverse=True).

2. **Aleatoriedad (Randomized)**
Para evitar el determinismo de la estrategia voraz pura, el código restringe la selección utilizando la variable rcl_size=5. El algoritmo toma los 5 mejores candidatos únicos de la lista ordenada previamente y los introduce en la lista rcl (Lista Restringida de Candidatos). La aleatoriedad se aplica inmediatamente después ejecutando la instrucción eleccion = random.choice(rcl), seleccionando al azar uno de esos mejores cruces.

3. **Adaptación (Adaptive)**
El proceso es adaptativo porque el entorno se actualiza dinámicamente dentro del bucle while equipos_disp:. Cada vez que se asigna un partido, su horario se guarda en la lista horarios_asignados. En la siguiente iteración, al reevaluar a los candidatos restantes, el código calcula dinámicamente el solapamiento usando coinc = min(horarios_asignados.count(h), 8). Esto hace que el beneficio de los cruces restantes se adapte instantáneamente (bajando su valor) si intentan ocupar un horario que el algoritmo acaba de llenar.

4. **Búsqueda (Search)**
Se ha implementado mediante la función busqueda_local_avanzada (y busqueda_local_df). Una vez que la fase constructiva entrega una solucion_inicial, se entra en un bucle while mejora: que explora el vecindario de la solución aplicando sistemáticamente tres movimientos:

    - Intercambio de horarios entre dos partidos existentes.

    - Movimiento de un partido a un horario distinto.

    - Intercambio de equipos entre dos partidos para romper bloqueos (ej. buscar cruces directos entre equipos "A").
    
      Si el valor devuelto por evaluar_solucion_completa mejora el mejor_aud, se acepta el movimiento como nuevo óptimo local.


5. **Procedimiento / Rearranque (Procedure)**
El mecanismo de rearranque (o multiarranque) se ha conseguido en la sección "EJECUCIÓN DEL BUCLE GRASP" introduciendo ambas fases dentro de un gran bucle principal: for _ in range(500): (o range(50) en la versión de DataFrames puros). Cada paso del bucle crea una solución desde cero, la optimiza y la compara con la variable max_audiencia_global. Esto asegura la exploración masiva del espacio de soluciones sin quedar atrapados en el primer resultado local que se genere.

---
####Evolución y Rendimiento (Pandas vs Diccionarios):

Se desarrolló una primera versión operando directamente sobre los DataFrames de Pandas. Sin embargo, debido al alto coste computacional de buscar celdas individuales millones de veces en la búsqueda local (tardando varios minutos), se rediseñó la arquitectura. La versión final transforma los DataFrames en diccionarios nativos de Python (*O(1)* ) antes del bucle, reduciendo el tiempo de ejecución a apenas 2 segundos para 500 iteraciones.

In [ ]:
# =================================================================
# 1. OPTIMIZACIÓN Y DICCIONARIOS (Carga instantánea)
# =================================================================
dict_categorias = dict(zip(df_equipos['Equipo'], df_equipos['Categoria']))

dict_base = {}
for _, row in df_audiencia_base.iterrows():
    dict_base[(row['Categoria_1'], row['Categoria_2'])] = row['Audiencia_Base_Millones']
    dict_base[(row['Categoria_2'], row['Categoria_1'])] = row['Audiencia_Base_Millones']

dict_pond = dict(zip(df_horarios.index, df_horarios['Ponderacion']))
dict_coinc = dict(zip(df_coincidencias['Coincidencias'], df_coincidencias['Reduccion_Porcentaje']))

def evaluar_solucion_completa(solucion):
    horarios_usados = [h for _, _, h in solucion]
    if 0 not in horarios_usados or 9 not in horarios_usados:
        return 0
    total = 0
    conteo = {h: horarios_usados.count(h) for h in set(horarios_usados)}
    for eq1, eq2, h in solucion:
        base = dict_base[(dict_categorias[eq1], dict_categorias[eq2])]
        coinc = min(conteo[h] - 1, 8)
        total += base * dict_pond[h] * (1 - dict_coinc[coinc])
    return total

# =================================================================
# 2. FASE CONSTRUCTIVA GRASP (Visión Global)
# =================================================================
def fase_constructiva_global(rcl_size=5):
    equipos_disp = list(df_equipos['Equipo'])
    sol_parcial = []
    horarios_asignados = []

    while equipos_disp:
        candidatos = []
        # Evaluamos el universo entero de cruces posibles
        for i in range(len(equipos_disp)):
            for j in range(i+1, len(equipos_disp)):
                eq1, eq2 = equipos_disp[i], equipos_disp[j]
                base = dict_base[(dict_categorias[eq1], dict_categorias[eq2])]

                for h in range(10):
                    # Estimamos la reducción
                    coinc = min(horarios_asignados.count(h), 8)
                    beneficio = base * dict_pond[h] * (1 - dict_coinc[coinc])
                    candidatos.append({'Eq1': eq1, 'Eq2': eq2, 'Horario': h, 'Beneficio': beneficio})

        candidatos.sort(key=lambda x: x['Beneficio'], reverse=True)

        # Filtramos para que la lista RCL tenga cruces variados y no se inunde de lo mismo
        rcl = []
        parejas_vistas = set()
        for c in candidatos:
            pareja = tuple(sorted([c['Eq1'], c['Eq2']]))
            if pareja not in parejas_vistas:
                rcl.append(c)
                parejas_vistas.add(pareja)
            if len(rcl) == rcl_size:
                break

        eleccion = random.choice(rcl)

        sol_parcial.append((eleccion['Eq1'], eleccion['Eq2'], eleccion['Horario']))
        horarios_asignados.append(eleccion['Horario'])
        equipos_disp.remove(eleccion['Eq1'])
        equipos_disp.remove(eleccion['Eq2'])

    if 0 not in horarios_asignados: sol_parcial[0] = (sol_parcial[0][0], sol_parcial[0][1], 0)
    if 9 not in horarios_asignados: sol_parcial[1] = (sol_parcial[1][0], sol_parcial[1][1], 9)
    return sol_parcial

# =================================================================
# 3. BÚSQUEDA LOCAL (Vecindario Avanzado)
# =================================================================
def busqueda_local_avanzada(solucion_inicial):
    mejor_sol = solucion_inicial.copy()
    mejor_aud = evaluar_solucion_completa(mejor_sol)
    mejora = True

    while mejora:
        mejora = False

        # Intercambiar horarios entre dos partidos (Ordenación perfecta)
        for i in range(10):
            for j in range(i+1, 10):
                vecino = mejor_sol.copy()
                vecino[i] = (mejor_sol[i][0], mejor_sol[i][1], mejor_sol[j][2])
                vecino[j] = (mejor_sol[j][0], mejor_sol[j][1], mejor_sol[i][2])
                aud_vecino = evaluar_solucion_completa(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # Cambiar un partido a un horario libre
        for i in range(10):
            h_actual = mejor_sol[i][2]
            for h_nuevo in range(10):
                if h_nuevo != h_actual:
                    vecino = mejor_sol.copy()
                    vecino[i] = (vecino[i][0], vecino[i][1], h_nuevo)
                    aud_vecino = evaluar_solucion_completa(vecino)
                    if aud_vecino > mejor_aud:
                        mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # Intercambiar equipos para romper bloqueos
        for i in range(10):
            for j in range(i+1, 10):
                eq1_i, eq2_i, h_i = mejor_sol[i]
                eq1_j, eq2_j, h_j = mejor_sol[j]

                vecino = mejor_sol.copy()
                vecino[i] = (eq1_j, eq2_i, h_i)
                vecino[j] = (eq1_i, eq2_j, h_j)
                aud_vecino = evaluar_solucion_completa(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break

    return mejor_sol, mejor_aud

# =================================================================
# 4. EJECUCIÓN (500 iteraciones)
# =================================================================
random.seed(123)
mejor_global = None
max_audiencia_global = 0

print("Explorando el espacio de soluciones con GRASP Avanzado...")

# 500 iteraciones ofrecen una alta probabilidad de encontrar una solución cuasi-óptima
for _ in range(500):
    sol_construida = fase_constructiva_global(rcl_size=5)
    sol_optimizada, audiencia_local = busqueda_local_avanzada(sol_construida)

    if audiencia_local > max_audiencia_global:
        max_audiencia_global = audiencia_local
        mejor_global = sol_optimizada

# Formatear la tabla
resultados = []
for eq1, eq2, h in mejor_global:
    cat1, cat2 = sorted([dict_categorias[eq1], dict_categorias[eq2]])
    base = dict_base[(cat1, cat2)]
    pond = dict_pond[h]
    resultados.append({
        'Enfrentamiento': f"{eq1} vs {eq2}",
        'Categorias': f"{cat1} vs {cat2}",
        'Audiencia_Base': base,
        'Horario_Asignado': df_horarios.loc[h, 'Horario'],
        'Ponderacion': pond,
        'Audiencia_Millones': round(base * pond, 4)
    })

df_final = pd.DataFrame(resultados).sort_values(by='Audiencia_Millones', ascending=False)

print(f"\n=========================================")
print(f"RESULTADO ALCANZADO")
print(f"Audiencia Total: {max_audiencia_global:.4f} Millones")
print(f"=========================================\n")
print(df_final.to_string(index=False))

Explorando el espacio de soluciones con GRASP Avanzado...

RESULTADO ALCANZADO
Audiencia Total: 7.2230 Millones

            Enfrentamiento Categorias  Audiencia_Base Horario_Asignado  Ponderacion  Audiencia_Millones
Real Madrid vs R. Sociedad     A vs A            2.00              S20         1.00              2.0000
     Barcelona vs Athletic     A vs B            1.30              D20         1.00              1.3000
        Atlético vs Getafe     B vs B            0.90              D18         0.85              0.7650
     Villarreal vs Sevilla     B vs B            0.90              S18         0.80              0.7200
           Alavés vs Betis     B vs B            0.90              D16         0.75              0.6750
      Mallorca vs Espanyol     B vs C            0.75              S16         0.70              0.5250
         Eibar vs Valencia     B vs C            0.75              S12         0.55              0.4125
          Celta vs Osasuna     B vs C            0.75  

In [ ]:
# =================================================================
# 2. FUNCIONES DE EVALUACIÓN (Consultando directamente a los DF)
# =================================================================
def obtener_categoria(eq):
    return df_equipos.loc[df_equipos['Equipo'] == eq, 'Categoria'].values[0]

def obtener_base(eq1, eq2):
    cat1, cat2 = sorted([obtener_categoria(eq1), obtener_categoria(eq2)])
    return df_audiencia_base[(df_audiencia_base['Categoria_1'] == cat1) &
                             (df_audiencia_base['Categoria_2'] == cat2)]['Audiencia_Base_Millones'].values[0]

def evaluar_solucion_completa_df(solucion):
    horarios_usados = [h for _, _, h in solucion]
    if 0 not in horarios_usados or 9 not in horarios_usados:
        return 0

    total = 0
    conteo = {h: horarios_usados.count(h) for h in set(horarios_usados)}

    for eq1, eq2, h in solucion:
        base = obtener_base(eq1, eq2)
        pond = df_horarios.loc[h, 'Ponderacion']
        coinc = min(conteo[h] - 1, 8)
        red = df_coincidencias.loc[df_coincidencias['Coincidencias'] == coinc, 'Reduccion_Porcentaje'].values[0]
        total += base * pond * (1 - red)
    return total

# =================================================================
# 3. FASE CONSTRUCTIVA GRASP CON DATAFRAMES
# =================================================================
def fase_constructiva_df(rcl_size=5):
    equipos_disp = list(df_equipos['Equipo'])
    sol_parcial = []
    horarios_asignados = []

    while equipos_disp:
        candidatos = []
        # Evaluamos cruces posibles (búsqueda global para evitar la trampa voraz)
        for i in range(len(equipos_disp)):
            for j in range(i+1, len(equipos_disp)):
                eq1, eq2 = equipos_disp[i], equipos_disp[j]
                base = obtener_base(eq1, eq2)

                for h in range(10):
                    pond = df_horarios.loc[h, 'Ponderacion']
                    coinc = min(horarios_asignados.count(h), 8)
                    red = df_coincidencias.loc[df_coincidencias['Coincidencias'] == coinc, 'Reduccion_Porcentaje'].values[0]
                    beneficio = base * pond * (1 - red)
                    candidatos.append({'Eq1': eq1, 'Eq2': eq2, 'Horario': h, 'Beneficio': beneficio})

        candidatos.sort(key=lambda x: x['Beneficio'], reverse=True)

        # Lista restringida (RCL) filtrando duplicados
        rcl = []
        parejas_vistas = set()
        for c in candidatos:
            pareja = tuple(sorted([c['Eq1'], c['Eq2']]))
            if pareja not in parejas_vistas:
                rcl.append(c)
                parejas_vistas.add(pareja)
            if len(rcl) == rcl_size:
                break

        eleccion = random.choice(rcl)

        sol_parcial.append((eleccion['Eq1'], eleccion['Eq2'], eleccion['Horario']))
        horarios_asignados.append(eleccion['Horario'])
        equipos_disp.remove(eleccion['Eq1'])
        equipos_disp.remove(eleccion['Eq2'])

    # Obligaciones: Viernes (0) y Lunes (9)
    if 0 not in horarios_asignados: sol_parcial[0] = (sol_parcial[0][0], sol_parcial[0][1], 0)
    if 9 not in horarios_asignados: sol_parcial[1] = (sol_parcial[1][0], sol_parcial[1][1], 9)
    return sol_parcial

# =================================================================
# 4. BÚSQUEDA LOCAL AVANZADA CON DATAFRAMES
# =================================================================
def busqueda_local_df(solucion_inicial):
    mejor_sol = solucion_inicial.copy()
    mejor_aud = evaluar_solucion_completa_df(mejor_sol)
    mejora = True

    while mejora:
        mejora = False

        # Intercambiar horarios entre dos partidos
        for i in range(10):
            for j in range(i+1, 10):
                vecino = mejor_sol.copy()
                vecino[i] = (mejor_sol[i][0], mejor_sol[i][1], mejor_sol[j][2])
                vecino[j] = (mejor_sol[j][0], mejor_sol[j][1], mejor_sol[i][2])
                aud_vecino = evaluar_solucion_completa_df(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # Mover un partido a otro horario libre
        for i in range(10):
            h_actual = mejor_sol[i][2]
            for h_nuevo in range(10):
                if h_nuevo != h_actual:
                    vecino = mejor_sol.copy()
                    vecino[i] = (vecino[i][0], vecino[i][1], h_nuevo)
                    aud_vecino = evaluar_solucion_completa_df(vecino)
                    if aud_vecino > mejor_aud:
                        mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break
        if mejora: continue

        # Intercambiar equipos para encontrar el Clásico
        for i in range(10):
            for j in range(i+1, 10):
                eq1_i, eq2_i, h_i = mejor_sol[i]
                eq1_j, eq2_j, h_j = mejor_sol[j]

                vecino = mejor_sol.copy()
                vecino[i] = (eq1_j, eq2_i, h_i)
                vecino[j] = (eq1_i, eq2_j, h_j)
                aud_vecino = evaluar_solucion_completa_df(vecino)
                if aud_vecino > mejor_aud:
                    mejor_aud = aud_vecino; mejor_sol = vecino; mejora = True; break
            if mejora: break

    return mejor_sol, mejor_aud

# =================================================================
# 5. EJECUCIÓN DEL BUCLE GRASP
# =================================================================
random.seed(42)
mejor_global = None
max_audiencia_global = 0

print("Ejecutando GRASP Avanzado (Version DataFrames Puros)...")

# Reducimos a 50 iteraciones para no saturar el tiempo de espera por culpa de Pandas
for _ in range(50):
    sol_construida = fase_constructiva_df(rcl_size=5)
    sol_optimizada, audiencia_local = busqueda_local_df(sol_construida)

    if audiencia_local > max_audiencia_global:
        max_audiencia_global = audiencia_local
        mejor_global = sol_optimizada

# Preparar tabla final
resultados = []
for eq1, eq2, h in mejor_global:
    cat1, cat2 = sorted([obtener_categoria(eq1), obtener_categoria(eq2)])
    base = obtener_base(eq1, eq2)
    pond = df_horarios.loc[h, 'Ponderacion']
    resultados.append({
        'Enfrentamiento': f"{eq1} vs {eq2}",
        'Categorias': f"{cat1} vs {cat2}",
        'Horario_Asignado': df_horarios.loc[h, 'Horario'],
        'Ponderacion': pond,
        'Audiencia_Millones': round(base * pond, 4)
    })

df_final = pd.DataFrame(resultados).sort_values(by='Audiencia_Millones', ascending=False)

print(f"\n=========================================")
print(f"RESULTADO ALCANZADO CON DATAFRAMES")
print(f"Audiencia Total: {max_audiencia_global:.4f} Millones")
print(f"=========================================\n")
print(df_final.to_string(index=False))

Ejecutando GRASP Avanzado (Version DataFrames Puros)...

RESULTADO ALCANZADO CON DATAFRAMES
Audiencia Total: 7.2230 Millones

            Enfrentamiento Categorias Horario_Asignado  Ponderacion  Audiencia_Millones
Real Madrid vs R. Sociedad     A vs A              S20         1.00              2.0000
        Barcelona vs Celta     A vs B              D20         1.00              1.3000
           Betis vs Getafe     B vs B              D18         0.85              0.7650
       Athletic vs Levante     B vs B              S18         0.80              0.7200
     Villarreal vs Sevilla     B vs B              D16         0.75              0.6750
      Mallorca vs Atlético     B vs C              S16         0.70              0.5250
         Osasuna vs Alavés     B vs C              S12         0.55              0.4125
       Valencia vs Granada     B vs C              D12         0.45              0.3375
    Espanyol vs Valladolid     B vs C              L20         0.40              0

[**Nota sobre el uso de IAG:** Todas las explicaciones y el desarrollo de este documento han sido redactados por el autor. Se ha empleado IAG de forma puntual y exclusiva para ayudar a resumir y estructurar algunas partes específicas del texto. El código, diseño algorítmico y análisis matemático son de autoría propia.]